In [1]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
import os

# 한글 폰트 설정
font_path = r'C:\Windows\Fonts\malgun.ttf'
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams['font.family'] = font_prop.get_name()
plt.rcParams['axes.unicode_minus'] = False

# 경로 설정 (workflows/ 기준 → 상위 디렉토리)
BASE_DIR    = os.path.dirname(os.path.abspath('__file__'))
TASK_DIR    = os.path.dirname(BASE_DIR)  # 03_Visualization_Slide_Deck/
CSV_PATH    = os.path.join(TASK_DIR, 'resources', 'monthly_revenue.csv')
OUTPUT_DIR  = os.path.join(TASK_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 지역별 색상 팔레트
COLORS = {
    'North': '#2E86AB',
    'South': '#A23B72',
    'East':  '#F18F01',
    'West':  '#C73E1D',
}
REGION_KR = {'North': '북부', 'South': '남부', 'East': '동부', 'West': '서부'}

MONTH_ORDER = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
MONTH_KR    = ['1월','2월','3월','4월','5월','6월',
               '7월','8월','9월','10월','11월','12월']

df = pd.read_csv(CSV_PATH)
df['month_name'] = pd.Categorical(df['month_name'], categories=MONTH_ORDER, ordered=True)
df['month_kr']   = df['month'].apply(lambda m: MONTH_KR[m-1])
df['region_kr']  = df['region'].map(REGION_KR)
df = df.sort_values(['month', 'region'])
print('데이터 로드 완료:', df.shape)
df.head()

데이터 로드 완료: (48, 9)


,month,month_name,region,revenue,units_sold,avg_order_value,new_customers,month_kr,region_kr
2,1,January,East,61000,210,290.48,14,1월,동부
0,1,January,North,52000,180,288.89,12,1월,북부
1,1,January,South,48500,165,293.94,10,1월,남부
3,1,January,West,55500,190,292.11,11,1월,서부
6,2,February,East,58500,200,292.50,13,2월,동부


In [2]:
# 공통 스타일 적용 함수
def apply_style(ax, title, xlabel='', ylabel=''):
    ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

In [3]:
# ── 차트 1: 지역별 월별 매출 추이 꺾은선 차트 ──────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

for region, color in COLORS.items():
    sub = df[df['region'] == region].sort_values('month')
    ax.plot(sub['month_kr'], sub['revenue'],
            marker='o', label=REGION_KR[region],
            color=color, linewidth=2.2, markersize=6)

ax.set_xticks(range(len(MONTH_KR)))
ax.set_xticklabels(MONTH_KR, rotation=45, ha='right', fontsize=9)
apply_style(ax, '2024년 지역별 월별 매출 추이', xlabel='월', ylabel='매출 (USD)')
ax.legend(title='지역', framealpha=0.9)
fig.tight_layout()

chart1_path = os.path.join(OUTPUT_DIR, 'chart1_line_trend.png')
fig.savefig(chart1_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'저장 완료: {chart1_path}')

저장 완료: G:\내 드라이브\강의자료\Vibe_Coding\Datascience\Test\03_Visualization_Slide_Deck\output\chart1_line_trend.png


In [4]:
# ── 차트 2: 지역별 연간 총 매출 막대 차트 ─────────────────────────────
annual = df.groupby('region')['revenue'].sum().reset_index()
annual['region_kr'] = annual['region'].map(REGION_KR)
annual = annual.sort_values('revenue', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(annual['region_kr'], annual['revenue'],
              color=[COLORS[r] for r in annual['region']],
              width=0.55, edgecolor='white')

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 15000,
            f'${h/1e6:.2f}M', ha='center', va='bottom', fontsize=10, fontweight='bold')

apply_style(ax, '2024년 지역별 연간 총 매출', ylabel='총 매출 (USD)')
ax.set_ylim(0, annual['revenue'].max() * 1.15)
fig.tight_layout()

chart2_path = os.path.join(OUTPUT_DIR, 'chart2_bar_annual.png')
fig.savefig(chart2_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'저장 완료: {chart2_path}')

저장 완료: G:\내 드라이브\강의자료\Vibe_Coding\Datascience\Test\03_Visualization_Slide_Deck\output\chart2_bar_annual.png


In [5]:
# ── 차트 3: 월별 × 지역별 매출 히트맵 ────────────────────────────────
pivot = df.pivot_table(index='month_name', columns='region', values='revenue')
pivot = pivot.reindex(MONTH_ORDER)
pivot.index = MONTH_KR
pivot.columns = [REGION_KR[c] for c in pivot.columns]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(pivot / 1000, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': '매출 (천 USD)'})
ax.set_title('2024년 월별 × 지역별 매출 히트맵', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('지역', fontsize=11)
ax.set_ylabel('월', fontsize=11)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)
fig.tight_layout()

chart3_path = os.path.join(OUTPUT_DIR, 'chart3_heatmap.png')
fig.savefig(chart3_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'저장 완료: {chart3_path}')

C:\Users\user\AppData\Local\Temp\ipykernel_12916\169452953.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = df.pivot_table(index='month_name', columns='region', values='revenue')


저장 완료: G:\내 드라이브\강의자료\Vibe_Coding\Datascience\Test\03_Visualization_Slide_Deck\output\chart3_heatmap.png


In [6]:
# ── 차트 4: 총 매출 기준 상위 월 요약 막대 차트 ───────────────────────
monthly_total = df.groupby(['month', 'month_kr'])['revenue'].sum().reset_index()
monthly_total = monthly_total.sort_values('revenue', ascending=False)

cmap = plt.colormaps.get_cmap('RdYlGn')
bar_colors = [cmap(i / (len(monthly_total)-1)) for i in range(len(monthly_total))][::-1]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(len(monthly_total)), monthly_total['revenue'],
              color=bar_colors, edgecolor='white', width=0.65)
ax.set_xticks(range(len(monthly_total)))
ax.set_xticklabels(monthly_total['month_kr'], rotation=40, ha='right', fontsize=9)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 8000,
            f'${h/1e6:.2f}M', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

apply_style(ax, '전체 지역 총 매출 기준 상위 월', ylabel='총 매출 (USD)')
ax.set_ylim(0, monthly_total['revenue'].max() * 1.13)
fig.tight_layout()

chart4_path = os.path.join(OUTPUT_DIR, 'chart4_top_months.png')
fig.savefig(chart4_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'저장 완료: {chart4_path}')

저장 완료: G:\내 드라이브\강의자료\Vibe_Coding\Datascience\Test\03_Visualization_Slide_Deck\output\chart4_top_months.png


In [7]:
# ── PowerPoint 프레젠테이션 생성 ──────────────────────────────────────
prs = Presentation()
prs.slide_width  = Inches(13.33)
prs.slide_height = Inches(7.5)

DARK_BLUE  = RGBColor(0x1A, 0x37, 0x5E)
MID_BLUE   = RGBColor(0x2E, 0x86, 0xAB)
WHITE      = RGBColor(0xFF, 0xFF, 0xFF)
LIGHT_GRAY = RGBColor(0xF4, 0xF6, 0xF9)

def add_text_box(slide, text, left, top, width, height,
                 font_size=14, bold=False, color=None,
                 align=PP_ALIGN.LEFT, bg_color=None):
    txBox = slide.shapes.add_textbox(left, top, width, height)
    if bg_color:
        txBox.fill.solid()
        txBox.fill.fore_color.rgb = bg_color
    tf = txBox.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.alignment = align
    run = p.add_run()
    run.text = text
    run.font.size = Pt(font_size)
    run.font.bold = bold
    if color:
        run.font.color.rgb = color
    return txBox

def fill_slide_bg(slide, color):
    fill = slide.background.fill
    fill.solid()
    fill.fore_color.rgb = color

# 슬라이드 1: 타이틀 슬라이드
slide = prs.slides.add_slide(prs.slide_layouts[6])
fill_slide_bg(slide, DARK_BLUE)
add_text_box(slide, '2024 매출 성과 리뷰',
             Inches(1), Inches(2.5), Inches(11.33), Inches(1.2),
             font_size=40, bold=True, color=WHITE, align=PP_ALIGN.CENTER)
add_text_box(slide, '지역별 매출 분석 · 2024년 1월 – 12월',
             Inches(1), Inches(3.9), Inches(11.33), Inches(0.6),
             font_size=18, color=MID_BLUE, align=PP_ALIGN.CENTER)
add_text_box(slide, '데이터 분석팀',
             Inches(1), Inches(6.3), Inches(11.33), Inches(0.5),
             font_size=12, color=RGBColor(0xCC,0xCC,0xCC), align=PP_ALIGN.CENTER)
print('슬라이드 1: 타이틀 생성')

슬라이드 1: 타이틀 생성


In [8]:
# 슬라이드 2-5: 차트 슬라이드 (차트별 인사이트 포함)
chart_slides = [
    (chart1_path,
     '지역별 월별 매출 추이',
     ('11월과 12월에 전 지역에서 매출이 최고조에 달하며, 연말 성수기 수요가 큰 영향을 미치고 있습니다. '
      '동부 지역이 매달 가장 높은 매출을 기록하며 꾸준한 1위를 유지했습니다. '
      '모든 지역이 유사한 계절 패턴을 보이며, 중반부 정체 후 연말에 급등하는 추세를 보였습니다.')),
    (chart2_path,
     '지역별 연간 총 매출',
     ('동부 지역이 약 114만 달러로 가장 높은 연간 매출을 기록하며 모든 지역을 앞섰습니다. '
      '서부와 남부가 각각 2위와 3위를 차지했으며, 북부는 약 87만 달러로 최하위였습니다. '
      '동부와 북부의 약 27만 달러 격차는 지역 간 성과 불균형이 상당함을 보여줍니다.')),
    (chart3_path,
     '월별 × 지역별 매출 히트맵',
     ('11월과 12월이 가장 진한 색조로 나타나며 4분기가 성수기임을 확인할 수 있습니다. '
      '1월과 2월은 전 지역에서 가장 낮은 매출을 기록하는 비수기입니다. '
      '동부 지역이 모든 셀에서 가장 높은 수치를 보이며 최고 성과 지역임을 재확인합니다.')),
    (chart4_path,
     '전체 지역 총 매출 기준 상위 월',
     ('12월이 약 64만 달러로 1위, 11월이 약 58만 달러로 2위를 차지했습니다. '
      '이 두 달만으로 연간 총 매출의 약 30%를 차지하는 중요한 성수기입니다. '
      '5–8월(5월~8월)은 안정적인 중간 수준을 형성하며 여름 수요가 꾸준함을 보여줍니다.')),
]

for i, (img_path, title, insight) in enumerate(chart_slides, start=2):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    fill_slide_bg(slide, LIGHT_GRAY)

    # 헤더 배경 바
    header = slide.shapes.add_shape(1, Inches(0), Inches(0),
                                    prs.slide_width, Inches(0.7))
    header.fill.solid()
    header.fill.fore_color.rgb = DARK_BLUE
    header.line.fill.background()

    add_text_box(slide, title,
                 Inches(0.3), Inches(0.05), Inches(12), Inches(0.6),
                 font_size=20, bold=True, color=WHITE)
    slide.shapes.add_picture(img_path, Inches(0.3), Inches(0.8),
                             Inches(12.7), Inches(5.0))

    # 인사이트 박스
    insight_box = slide.shapes.add_shape(1, Inches(0), Inches(5.9),
                                         prs.slide_width, Inches(1.6))
    insight_box.fill.solid()
    insight_box.fill.fore_color.rgb = DARK_BLUE
    insight_box.line.fill.background()

    add_text_box(slide, insight,
                 Inches(0.4), Inches(5.97), Inches(12.5), Inches(1.4),
                 font_size=12, color=WHITE)
    print(f'슬라이드 {i}: {title}')

슬라이드 2: 지역별 월별 매출 추이
슬라이드 3: 지역별 연간 총 매출
슬라이드 4: 월별 × 지역별 매출 히트맵
슬라이드 5: 전체 지역 총 매출 기준 상위 월


In [9]:
# 슬라이드 6: 핵심 시사점 요약 슬라이드
slide = prs.slides.add_slide(prs.slide_layouts[6])
fill_slide_bg(slide, DARK_BLUE)
add_text_box(slide, '핵심 시사점',
             Inches(0.5), Inches(0.3), Inches(12), Inches(0.7),
             font_size=28, bold=True, color=WHITE)

takeaways = [
    ('1', '4분기가 연간 매출의 약 30%를 견인',
     '11월과 12월은 매출 성수기로, 사전에 재고·인력·마케팅 투자를 집중하여 '
     '연말 수요를 최대한 활용하는 전략이 필요합니다.'),
    ('2', '동부 지역이 전 기간 최고 성과 지역',
     '동부는 12개월 내내 월별·연간 매출 모두 1위를 유지했습니다. '
     '동부의 영업 전략과 고객 구성을 분석하여 타 지역에 적용할 성장 레버를 찾아야 합니다.'),
    ('3', '중반기 정체 구간에서 성장 기회 발굴 필요',
     '5~8월 전 지역에서 상대적으로 평탄한 성장세를 보입니다. '
     '중반기 타깃 프로모션이나 신제품 출시로 매출 곡선을 평탄화하고 4분기 의존도를 낮출 수 있습니다.'),
]

for i, (num, heading, body) in enumerate(takeaways):
    top = Inches(1.2 + i * 1.8)
    # 번호 원형 배경
    circle = slide.shapes.add_shape(9, Inches(0.4), top + Inches(0.15),
                                    Inches(0.55), Inches(0.55))
    circle.fill.solid()
    circle.fill.fore_color.rgb = MID_BLUE
    circle.line.fill.background()
    add_text_box(slide, num, Inches(0.4), top + Inches(0.12),
                 Inches(0.55), Inches(0.55),
                 font_size=16, bold=True, color=WHITE, align=PP_ALIGN.CENTER)
    add_text_box(slide, heading,
                 Inches(1.1), top, Inches(11), Inches(0.5),
                 font_size=15, bold=True, color=WHITE)
    add_text_box(slide, body,
                 Inches(1.1), top + Inches(0.5), Inches(11), Inches(1.0),
                 font_size=11, color=RGBColor(0xCC, 0xDD, 0xEE))

# PPTX 저장
pptx_path = os.path.join(OUTPUT_DIR, 'revenue_review.pptx')
prs.save(pptx_path)
print(f'슬라이드 6: 요약 슬라이드 생성')
print(f'\n저장 완료: {pptx_path}')
print('\n모든 작업 완료!')

슬라이드 6: 요약 슬라이드 생성

저장 완료: G:\내 드라이브\강의자료\Vibe_Coding\Datascience\Test\03_Visualization_Slide_Deck\output\revenue_review.pptx

모든 작업 완료!
